# Weighted Max-Cut with QAOA

## Qiskit Fall Fest 2026 — Phase 1, PS-O1

This notebook studies the weighted Max-Cut problem using the Quantum Approximate Optimization Algorithm (QAOA). It establishes a classical reference solution, examines the problem graph, and evaluates how processor connectivity and qubit capacity affect circuit compilation.

The notebook uses the project's graph and processor configuration files as inputs. Numerical results and compilation metrics should be reported only after the corresponding calculations have run successfully.

### Workflow
1. Load the project configuration and benchmark graph.
2. Verify the graph and compute its exact classical optimum.
3. Construct and optimize QAOA circuits at the required depths.
4. Evaluate circuit compilation against the configured processor topologies.
5. Compare circuit depth, gate counts, and routing overhead where compilation is feasible.

In [ ]:
import json
from pathlib import Path

import networkx as nx

ROOT = Path.cwd()

if not (ROOT / "src").is_dir() or not (ROOT / "data").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the project root, where the src/ and data/ folders are located."
    )

from src.processors import load_processor, get_undirected_edges

print("Project root:", ROOT.resolve())
print("Project source and data directories found.")

## 1. Processor configurations

The processor definitions are loaded from the project configuration rather than duplicated in this notebook. This keeps the reported qubit counts and connectivity consistent with the files used by the compilation code.

A topology should be treated as authoritative only after it has been checked against the organizer's specifications. Results from an unverified configuration should not be presented as official hardware comparisons.

In [ ]:
proc_a = load_processor("processor_A")
proc_b = load_processor("processor_B")

for processor in (proc_a, proc_b):
    print(f"{processor['name']}: {processor['num_qubits']} physical qubits")
    print("Undirected connectivity:", get_undirected_edges(processor))
    print()

## 2. Benchmark graph and exact classical solution

The team-selected benchmark is a weighted graph with six vertices and nine edges. The exact classical optimum is obtained by exhaustively evaluating every binary partition of the vertices.

For a bitstring representing a partition, an edge contributes its weight when its endpoints belong to different sides. The objective is to maximize the sum of these crossing-edge weights.

The calculation below derives the optimum from the graph data rather than relying solely on a stored expected value.

In [ ]:
graph_path = ROOT / "data" / "weighted_maxcut_g6.json"
graph_data = json.loads(graph_path.read_text(encoding="utf-8"))

n = graph_data["num_vertices"]
edges = graph_data["edges"]

G = nx.Graph()
G.add_nodes_from(range(n))

for edge in edges:
    if isinstance(edge, dict):
        u, v, weight = edge["u"], edge["v"], edge["weight"]
    else:
        u, v, weight = edge[0], edge[1], edge[2]
    G.add_edge(int(u), int(v), weight=float(weight))

if n != 6:
    raise ValueError(f"Expected the six-vertex benchmark, found {n} vertices.")
if G.number_of_edges() != len(edges):
    raise ValueError("The graph data contains duplicate edges or inconsistent edge entries.")
if any(weight <= 0 for _, _, weight in G.edges(data="weight")):
    raise ValueError("All benchmark edge weights must be positive.")

def cut_weight(bitstring, graph):
    return sum(
        data["weight"]
        for u, v, data in graph.edges(data=True)
        if bitstring[u] != bitstring[v]
    )

best_value = float("-inf")
optimal_bitstrings = []

for integer in range(1 << n):
    bits = tuple((integer >> i) & 1 for i in range(n))
    value = cut_weight(bits, G)
    if value > best_value:
        best_value = value
        optimal_bitstrings = [bits]
    elif value == best_value:
        optimal_bitstrings.append(bits)

print("Benchmark:", graph_data.get("title", graph_path.name))
print("Vertices:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Total edge weight:", sum(data["weight"] for _, _, data in G.edges(data=True)))
print("Exact maximum cut:", best_value)
print("Optimal bitstrings (vertex order 0 to n-1):")
for bits in optimal_bitstrings:
    print("".join(map(str, bits)))

## 3. Logical qubit capacity

The benchmark uses six logical qubits. A processor must have enough physical qubits to accommodate the circuit before connectivity and routing can be evaluated.

A capacity failure is reported explicitly; it is not treated as a successful compilation. Passing the capacity check only establishes that the number of physical qubits is sufficient. It does not guarantee that transpilation will succeed or that the resulting circuit will have a particular SWAP count.

In [ ]:
logical_qubits = G.number_of_nodes()

for processor in (proc_a, proc_b):
    physical_qubits = processor["num_qubits"]
    feasible = logical_qubits <= physical_qubits
    status = "Capacity sufficient" if feasible else "Insufficient physical qubits"
    print(
        f"{processor['name']}: {logical_qubits} logical qubits, "
        f"{physical_qubits} physical qubits — {status}"
    )

## 4. QAOA optimization

QAOA alternates problem-dependent cost operations with a mixing operation. A classical optimizer searches for variational angles that improve the expected cut value.

The project implementation should be used to run the required circuit depths, optimizer, and measurement procedure. The exact classical optimum computed above provides the reference for evaluating the quality of the QAOA result.

For a measured best cut value $C_{\mathrm{best}}$ and exact optimum $C^*$, the approximation ratio is $C_{\mathrm{best}}/C^*$ when $C^*>0$. The probability of an optimal bitstring is calculated from the measurement counts by summing the counts of all optimal bitstrings and dividing by the total number of shots.

This notebook does not assume that optimization has succeeded. Record optimizer settings, random seeds, shot counts, and actual returned measurement results when running the project's QAOA implementation.

## 5. Compilation and routing analysis

For each processor that can accommodate the logical circuit, compile the same logical circuit using the project's agreed basis gates and transpilation settings. Record the Qiskit version, optimization level, transpiler seed, initial and final layouts, compiled two-qubit gate count, circuit depth, and inserted SWAP count when these metrics are available.

Compare results only when the logical circuit and compilation settings are comparable. A processor that fails the capacity check should be reported as infeasible for this benchmark rather than assigned fabricated routing metrics.

The topology and basis-gate definitions must match the verified project configuration. Compilation results should be generated by running the transpiler, not entered manually.

## 6. Reproducibility and validation

Run the notebook from the project root after installing the dependencies specified by the repository. Before reporting results, run the project's tests and confirm that the graph, processor configurations, QAOA implementation, and routing metrics are consistent.

Example commands from the project root:

```bash
pip install -r requirements.txt
python -m pytest -q
```

Use the actual project runner documented in the repository to execute QAOA experiments. Save generated data and figures to the designated results directory, and document the settings used for each run.

No noisy-simulation results, hardware performance claims, or quantum-advantage claims are implied by this notebook. Such statements require corresponding experiments and evidence.